In [7]:
def cross_entropy(y):
    if len(y) == 0:
        return 0

    eps = 1e-9
    classes, counts = np.unique(y, return_counts=True)
    probs = counts / counts.sum()

    return -np.sum(probs * np.log2(probs + eps))


In [14]:
left_mask = X[:, 1] <= 35
left_mask

array([False, False, False, False, False, False])

In [2]:
def split_entropy(X, y, feature_index, threshold):
    left_mask = X[:, feature_index] <= threshold
    right_mask = X[:, feature_index] > threshold

    y_left = y[left_mask]
    y_right = y[right_mask]

    n = len(y)
    return (len(y_left) / n) * cross_entropy(y_left) + \
           (len(y_right) / n) * cross_entropy(y_right)


In [8]:
import numpy as np
# Dataset
X = np.array([
    [25, 40],
    [30, 60],
    [35, 65],
    [40, 70],
    [45, 80],
    [50, 90]
])
y = np.array([0, 0, 1, 1, 1, 1])

entropy = split_entropy(X, y, feature_index=0, threshold=35)
print("Cross Entropy:", entropy)


Cross Entropy: 0.45914791486320217


In [21]:
np.unique(X[:, 0])

array([25, 30, 35, 40, 45, 50])

In [15]:
def best_split(X, y):
    best_entropy = float("inf")
    best_feature = None
    best_threshold = None

    for feature in range(X.shape[1]):
        thresholds = np.unique(X[:, feature])
        for threshold in thresholds:
            entropy = split_entropy(X, y, feature, threshold)
            if entropy < best_entropy:
                best_entropy = entropy
                best_feature = feature
                best_threshold = threshold

    return best_feature, best_threshold, best_entropy


In [10]:
feature, threshold, entropy = best_split(X, y)
print("Best Feature:", feature)
print("Best Threshold:", threshold)
print("Best Cross Entropy:", entropy)


Best Feature: 0
Best Threshold: 30
Best Cross Entropy: -1.4426951595367387e-09


In [23]:
class DecisionTree:
    def __init__(self, max_depth=2):
        self.max_depth = max_depth
        self.tree = None

    def fit(self, X, y):
        self.tree = self._build_tree(X, y, depth=0)

    def _build_tree(self, X, y, depth):
        # If pure or max depth reached
        if len(set(y)) == 1 or depth == self.max_depth:
            return np.mean(y)

        feature, threshold, _ = best_split(X, y)
        if feature is None:
            return np.mean(y)

        left_mask = X[:, feature] <= threshold
        right_mask = X[:, feature] > threshold

        return {
            "feature": feature,
            "threshold": threshold,
            "left": self._build_tree(X[left_mask], y[left_mask], depth + 1),
            "right": self._build_tree(X[right_mask], y[right_mask], depth + 1)
        }

    def predict_one(self, x, node):
        if not isinstance(node, dict):
            return 1 if node >= 0.5 else 0

        if x[node["feature"]] <= node["threshold"]:
            return self.predict_one(x, node["left"])
        else:
            return self.predict_one(x, node["right"])

    def predict(self, X):
        return np.array([self.predict_one(x, self.tree) for x in X])


In [24]:
# Train tree
tree = DecisionTree(max_depth=2)
tree.fit(X, y)

# Predict
predictions = tree.predict(X)
print("Predictions:", predictions)


Predictions: [0 0 1 1 1 1]
